In [4]:
!pip install gymnasium -q
!pip install imageio -q
!pip install torch -q

In [2]:
import imageio
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import random

from collections import deque

In [5]:
env = gym.make('MountainCar-v0')

# Discretize state space
pos_space = np.linspace(env.observation_space.low[0], env.observation_space.high[0], 20)
vel_space = np.linspace(env.observation_space.low[1], env.observation_space.high[1], 20)

q_table = np.zeros((len(pos_space), len(vel_space), env.action_space.n))

learning_rate = 0.1
discount = 0.95
epochs = 2000
epsilon = 0.5
epsilon_decay = 0.998

def get_discrete_state(state):
    pos, vel = state
    pos_bin = np.digitize(pos, pos_space)
    vel_bin = np.digitize(vel, vel_space)
    return (pos_bin, vel_bin)

for epoch in range(epochs):
    state = get_discrete_state(env.reset()[0])
    terminated = False
    truncated = False

    if epoch % 100 == 0:
        print(f"Epoch: {epoch}")

    while not (terminated or truncated):
        # Epsilon-greedy policy
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state])

        new_state_continuous, reward, terminated, truncated, _ = env.step(action)
        new_state = get_discrete_state(new_state_continuous)

        if not (terminated or truncated):
            max_future_q = np.max(q_table[new_state])
            current_q = q_table[state + (action,)]
            new_q = current_q + learning_rate * (reward + discount * max_future_q - current_q)
            q_table[state + (action,)] = new_q
        else:
            q_table[state + (action,)] = 0

        state = new_state

    if epsilon > 0.05:
        epsilon *= epsilon_decay

print("Training finished!")
env.close()

Epoch: 0
Epoch: 100
Epoch: 200
Epoch: 300
Epoch: 400
Epoch: 500
Epoch: 600
Epoch: 700
Epoch: 800
Epoch: 900
Epoch: 1000
Epoch: 1100
Epoch: 1200
Epoch: 1300
Epoch: 1400
Epoch: 1500
Epoch: 1600
Epoch: 1700
Epoch: 1800
Epoch: 1900
Training finished!


In [6]:
video_env = gym.make('MountainCar-v0', render_mode='rgb_array')

frames = []
state = get_discrete_state(video_env.reset()[0])
done = False

while not done:
    action = np.argmax(q_table[state])
    obs, _, terminated, truncated, _ = video_env.step(action)
    state = get_discrete_state(obs)
    done = terminated or truncated

    frame = video_env.render()
    frames.append(frame)

video_env.close()

video_path = "mountaincar_RL.mp4"
imageio.mimsave(video_path, frames, fps=30)
print("Video saved:", video_path)

Video saved: mountaincar_RL.mp4


In [8]:
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

class ReplayBuffer:
    def __init__(self, size, batch_size):
        self.memory = deque(maxlen=size)
        self.batch = batch_size

    def add(self, s, a, r, ns, d):
        self.memory.append((s, a, r, ns, d))

    def sample(self):
        batch = random.sample(self.memory, self.batch)

        states = np.vstack([e[0] for e in batch])
        actions = np.vstack([e[1] for e in batch])
        rewards = np.vstack([e[2] for e in batch])
        next_states = np.vstack([e[3] for e in batch])
        dones = np.vstack([e[4] for e in batch]).astype(np.uint8)

        states = torch.tensor(states, dtype=torch.float32)
        actions = torch.tensor(actions, dtype=torch.long)
        rewards = torch.tensor(rewards, dtype=torch.float32)
        next_states = torch.tensor(next_states, dtype=torch.float32)
        dones = torch.tensor(dones, dtype=torch.float32)

        return states, actions, rewards, next_states, dones

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

env = gym.make('MountainCar-v0')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

q_local = QNetwork(state_size, action_size).to(device)
q_target = QNetwork(state_size, action_size).to(device)
optimizer = optim.Adam(q_local.parameters(), lr=0.0005)
q_target.load_state_dict(q_local.state_dict())

memory = ReplayBuffer(100000, 64)

def learn():
    states, actions, rewards, next_states, dones = memory.sample()
    states, actions, rewards, next_states, dones = states.to(device), actions.to(device), rewards.to(device), next_states.to(device), dones.to(device)

    best_actions = q_local(next_states).argmax(1).unsqueeze(1)
    q_targets = rewards + 0.99 * q_target(next_states).gather(1, best_actions) * (1 - dones)
    q_values = q_local(states).gather(1, actions)

    loss = nn.MSELoss()(q_values, q_targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    for tp, lp in zip(q_target.parameters(), q_local.parameters()):
        tp.data.copy_(0.01 * lp.data + 0.99 * tp.data)

episodes = 1200
epsilon = 1.0

for ep in range(1, episodes+1):
    state = env.reset()[0]
    score = 0

    for _ in range(1000):
        action = np.argmax(q_local(torch.tensor(state).float().to(device)).detach().cpu().numpy()) if random.random() > epsilon else random.randint(0,2)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Reward shaping
        pos, vel = next_state
        reward += 0.5 * abs(vel) + (pos + 0.5)

        memory.add(state, action, reward, next_state, done)
        state = next_state
        score += reward

        if len(memory.memory) > 64:
            learn()
        if done:
            break

    epsilon = max(0.01, epsilon * 0.997)
    if ep % 100 == 0:
        print(f"Episode {ep}")

env.close()
print("DQN Training Finished")

Using: cuda
Episode 100
Episode 200
Episode 300
Episode 400
Episode 500
Episode 600
Episode 700
Episode 800
Episode 900
Episode 1000
Episode 1100
Episode 1200
DQN Training Finished


In [9]:
render_env = gym.make('MountainCar-v0', render_mode='rgb_array')
frames = []
state = render_env.reset()[0]
done = False

while not done:
    state_t = torch.tensor(state).float().unsqueeze(0).to(device)
    action = q_local(state_t).argmax().item()
    next_state, _, terminated, truncated, _ = render_env.step(action)
    frames.append(render_env.render())
    state = next_state
    done = terminated or truncated

render_env.close()

video_path = "mountaincar_DRL.mp4"
imageio.mimsave(video_path, frames, fps=30)
print("Video saved:", video_path)

Video saved: mountaincar_DRL.mp4


In [ ]:
# from google.colab import files
# files.download(video_path)